In [ ]:
import numpy as np
import torch
from train_reinforce import BinPacking_Environment, get_action_from_idx
from PackingUtils import *
from ModelEMS import *
import datetime
import random
import json
import seaborn as sns
import matplotlib.pyplot as plt

In [44]:
def run_policy(item_list, model_filename, device_name, container_dim=(100, 100, 100), clip_num=50):
	device = torch.device(device_name)
	model = BPP_Model_EMS(num_placement=clip_num, batch_size=1, embed_size=128, feature_mlp_layers=[256, 128], feature_mlp_output=64).to(device)
	model.load_state_dict(torch.load(model_filename, map_location=device, weights_only=True))
	max_steps = 200
	used_bins_idx = []

	env = BinPacking_Environment(container_dim, item_list, clip_num=clip_num, device=device_name)
	step_cnt = 0
	done = False
	states, action_ids, rewards, actions = [], [], [], []
	model.eval()
	while not done:
		placement, item_info, height_map, feasibility_mask = env.get_state()
		ems_state, item_state, ems_feature, item_feature, logits_raw_, logits = model(
			placement,
			item_info,
			height_map,
			feasibility_mask
		)
		# print(feasibility_mask)
		# print(logits)
		# if torch.any(logits < 0) or torch.any(logits == torch.nan) or torch.any(logits == torch.inf):
		# 	print(f"logits < 0 exists")
		# 	print(logits)
		# print(logits)
		try:
			# action_idx = torch.multinomial(logits, 1).item()
			# action_idx = torch.topk(logits, 1, dim=-1).indices[0].item()
			action_idx = torch.argmax(logits, dim=-1).item()
			# print(logits)
		except Exception as e:
			print(f"Error: {e}")
			print(item_list)
			print(logits)
			print(logits_raw_)
			print("feasibility mask: ", feasibility_mask)
			print("placement: ", placement)
			print("item_info: ", item_info)
			print("EMS state: ", ems_state)
			print("Item state: ", item_state)
			print("height_map: ", height_map)

			print("EMS feature: ", ems_feature)
			print("Item feature: ", item_feature)
			print("step count: ", step_cnt)
			break
		# print(action_idx)
		item_size = item_info.cpu().detach().numpy()[0, 0].astype(int).tolist()
		# print(item_size, item_size[0])
		action = get_action_from_idx(placement, action_idx, clip_num, height_map=env.height_map, item_size=item_size, return_height=True)
		ax, ay, ar, height = action
		print(f"action: {action}")
		reward, done = env.step((ax, ay, ar), False)
		if reward == 0:
			print(f"skip this step: item {item_info.cpu().detach().numpy().tolist()}")
			env.item_ptr += 1
			if env.item_ptr >= len(item_list):
				done = True
		else:
			states.append((placement, item_state, height_map, feasibility_mask))
			rewards.append(reward)
			action_ids.append(action_idx)
			actions.append(action)
			if env.item_ptr not in used_bins_idx:
				used_bins_idx.append(env.item_ptr)
		if step_cnt >= max_steps and not done:
			done = True
			# rewards[-1] -= env.get_remaining_cnt() * 0.1
			break
		step_cnt += 1
	print(f"Used items: {env.used} / {len(env.items_list)}")
	return env, used_bins_idx, actions

In [ ]:
# container_dim = (52, 40, 17)
# container_dim = (100, 100, 100)
container_dim = (37, 26, 13)
items = np.load(f"dataset_map/item_list_{container_dim}.npy")
# print(items)
item_list_ = items.tolist()
print(item_list_)
item_list = []
for item in item_list_:
	item_list.append(np.array(item))
# train()
# random.shuffle(item_list)
# item_list = item_list[:15]
item_list.sort(key=lambda x: max(x[0], x[1]), reverse=True)
print(item_list)
# random.shuffle(item_list)


In [ ]:
env, used_bins_idx, actions = run_policy(item_list, "models/model_20250104_212255_.pth", "cpu", container_dim=container_dim)
print(used_bins_idx)
print(actions)

In [ ]:
height_map = env.height_map
state = None
if env.item_ptr == len(item_list):
	print("success")
else:
	coming_item = env.items_list[env.item_ptr]
	print(f"coming item: {coming_item}")
	states = env.get_state()
utilization = env.get_utilization()
print(f"utilization: {utilization}")

In [ ]:
fig = plt.figure(figsize=(6, 5), dpi=400)
sns.heatmap(height_map, square=True, center=50)
plt.title("Utilization: {:.2f}%".format(utilization*100))
plt.savefig("test_bin_packing.png")

In [49]:
if state:
	print(f"placement: {states[0]}")

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
# fig = plt.figure(figsize=(6, 5), dpi=100)

def draw_bins_3d(container_dim, positions, sizes, filename):
    """
    Draws a 3D figure of bins given their positions and sizes.

    Parameters:
    - positions: List of tuples (x, y, z) representing the bottom-left-front corner of each bin.
    - sizes: List of tuples (dx, dy, dz) representing the dimensions of each bin.
    """
    fig = plt.figure(figsize=(6, 5), dpi=400)
    ax = fig.add_subplot(111, projection='3d')
    nums = len(positions)
    ranges = [random.random() for _ in range(nums)]
    print(ranges)
    color_list = plt.cm.tab20b(ranges)

    for i, (pos, size) in enumerate(zip(positions, sizes)):
        x, y, z = pos
        dx, dy, dz = size
        facecolor = color_list[i]

        # Define the vertices of the cuboid
        vertices = [
            [x+1, y, z],
            [x-dx+1, y, z],
            [x-dx+1, y+dy, z],
            [x+1, y+dy, z],
            [x+1, y, z+dz],
            [x-dx+1, y, z+dz],
            [x-dx+1, y+dy, z+dz],
            [x+1, y+dy, z+dz],
        ]
        print(vertices)

        # Define the 12 edges of the cuboid
        edges = [
            [vertices[0], vertices[1], vertices[2], vertices[3]],  # Bottom face
            [vertices[4], vertices[5], vertices[6], vertices[7]],  # Top face
            [vertices[0], vertices[1], vertices[5], vertices[4]],  # Front face
            [vertices[2], vertices[3], vertices[7], vertices[6]],  # Back face
            [vertices[1], vertices[2], vertices[6], vertices[5]],  # Right face
            [vertices[0], vertices[3], vertices[7], vertices[4]],  # Left face
        ]

        # Add the cuboid to the plot
        ax.add_collection3d(Poly3DCollection(edges, alpha=1.0, edgecolor='k', facecolor=facecolor, linewidth=0.1))

    # Set axes limits and labels
    max_dim = max(max(pos[i] + size[i] for pos, size in zip(positions, sizes)) for i in range(3))
    ax.set_xlim(0, container_dim[0])
    ax.set_ylim(0, container_dim[1])
    ax.set_zlim(0, container_dim[2])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.view_init(10, 30)

    # plt.show()
    plt.savefig(filename)

# Example usage
# positions = [(0, 0, 0), (1, 1, 1), (2, 2, 2)]  # Bottom-left-front positions of bins
# sizes = [(1, 1, 1), (1, 2, 1), (2, 1, 2)]  # Dimensions (dx, dy, dz) of each bin

# draw_bins_3d(positions, sizes)

In [ ]:

# ax = fig.add_subplot(111, projection='3d')

bins = [item_list[idx-1].tolist() for idx in used_bins_idx]
positions = []
print(bins)
n_used_bins = len(used_bins_idx)
for i in range(n_used_bins):
	xpos, ypos, rotate, zpos = actions[i]
	if rotate == 1:
		bins[i] = [bins[i][1], bins[i][0], bins[i][2]]
	positions.append([xpos, ypos, zpos])
print(positions)
# ax.voxels(bins, edgecolor='k')
draw_bins_3d(positions, bins, "models/bin_packing_3d.png")